# Qwen3-8B HiRAG System for Kaggle Notebook

This notebook implements a complete HiRAG system using Qwen3-8B model and Hugging Face embedding models optimized for H100 GPU in notebook environments.

## Features
- Uses Qwen/Qwen3-8B (base model, not instruct)
- Uses all-mpnet-base-v2 embedding model optimized for H100
- Progress bars for all operations
- Timing for each step
- Asynchronous operations with notebook compatibility
- Processes knowledge graph using eval/datasets/cs/cs_unique_contexts.json
- Optimized for H100 GPU (80GB)

In [ ]:
# Install required dependencies
!pip install -q transformers torch sentence-transformers tiktoken tqdm

In [ ]:
# Import necessary modules
import os
import sys
import asyncio
import nest_asyncio

# Apply nest_asyncio to allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

print("Nest asyncio applied - ready for async operations in notebook")

In [ ]:
# Import the Qwen3 HiRAG system
from qwen3_notebook_system import run_notebook_pipeline, Qwen3HiRAGConfig

print("Qwen3 HiRAG system imported successfully!")

In [ ]:
# Verify GPU availability
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("GPU not available")
    
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Define your configuration parameters
config_params = {
    "llm_model_name": "Qwen/Qwen3-8B",  # Base Qwen3 8B model (not instruct)
    "embedding_model_name": "sentence-transformers/all-mpnet-base-v2",  # Better embedding model for H100
    "data_file": "/workspace/eval/datasets/cs/cs_unique_contexts.json",  # Knowledge graph data
    "num_contexts": 10,  # Number of contexts to process (adjust based on GPU memory)
    "queries": [
        "What are the key concepts in machine learning with Spark?",
        "Explain the architecture of a machine learning system",
        "How does collaborative filtering work in recommendation engines?"
    ]
}

print("Configuration parameters set:")
for key, value in config_params.items():
    if key == "queries":
        print(f"  {key}: {len(value)} queries")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Run the complete HiRAG pipeline
print("Starting Qwen3 HiRAG pipeline...")
print("This may take several minutes depending on model loading and data processing")

try:
    results = run_notebook_pipeline(
        llm_model_name=config_params["llm_model_name"],
        embedding_model_name=config_params["embedding_model_name"],
        data_file=config_params["data_file"],
        num_contexts=config_params["num_contexts"],
        queries=config_params["queries"]
    )
    
    print("\nPipeline completed successfully!")
    print(f"Received {len(results)} responses")
    
except Exception as e:
    print(f"Error occurred during pipeline execution: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
# Display results
if 'results' in locals():
    for i, (query, result) in enumerate(zip(config_params["queries"], results)):
        print(f"\n--- Query {i+1} ---")
        print(f"Question: {query}")
        print(f"Answer: {result}")
        print("-" * 50)

In [ ]:
# Clean up GPU memory if needed
import gc
import torch

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("GPU memory cleared")
    else:
        print("GPU not available, no cache to clear")

cleanup_gpu()

## Custom Configuration Example

You can easily modify the parameters to suit your needs:

In [ ]:
# Example with custom parameters
custom_results = run_notebook_pipeline(
    llm_model_name="Qwen/Qwen3-8B",  # Change this if you want to use a different model
    embedding_model_name="sentence-transformers/all-mpnet-base-v2",  # Change embedding model if needed
    data_file="/workspace/eval/datasets/cs/cs_unique_contexts.json",  # Path to your data file
    num_contexts=5,  # Adjust number of contexts based on your GPU memory
    queries=[
        "Explain neural network architectures",
        "What is the difference between supervised and unsupervised learning?",
        "How do decision trees work in machine learning?"
    ]
)

print(f"Custom pipeline completed with {len(custom_results)} results")